# 学习 LeRobotDataset：从 frame 到 batch

这个 Notebook 只学习数据读取，不训练模型。建议配合 `docs/lerobot_dataset_research/` 和 `docs/lerobot_learning/examples/dataset/` 一起阅读。

核心问题：

- `LeRobotDatasetMetadata` 里面有什么？
- `dataset[index]` 返回什么？
- 一个 frame、一个时间窗口、一个 batch 的形状如何变化？
- action chunk 为什么会有时间维度？

## 目标

执行完后，你应该能够解释下面这条链路：

```text
Hub 数据集
  -> LeRobotDatasetMetadata：目录和 schema
  -> LeRobotDataset[index]：以一个 frame 为中心的样本
  -> delta_timestamps：历史 observation 和未来 action
  -> DataLoader：把多个样本拼成 [B, ...]
```

## 设置

第一次打开时保持 `RUN_DATASET=False`，只阅读代码。完成环境准备并准备下载数据后，再把它改成 `True`，从头执行 Notebook。

运行前需要在仓库根目录的 LeRobot 环境中安装 Dataset 依赖，例如：

```bash
UV_PROJECT_ENVIRONMENT=.venv-libero uv sync --locked --extra evaluation --extra libero --extra smolvla
```

In [ ]:
# 安全开关：False 时不会访问 Hub，也不会下载视频。
RUN_DATASET = False

# 这里使用 LeRobot 文档推荐的 LIBERO 数据集。
DATASET_REPO = "lerobot/libero"
EPISODES = [0]          # 第一次只读第 0 个 episode，避免下载和解码太多数据
BATCH_SIZE = 2
HISTORY_STEPS = 3
ACTION_STEPS = 8

print(f"RUN_DATASET={RUN_DATASET}")
print(f"DATASET_REPO={DATASET_REPO}")
print(f"EPISODES={EPISODES}")

## 1. 读取 metadata：先看目录，再看数据

`LeRobotDatasetMetadata` 读取 episode 数、frame 数、fps、相机 key、feature schema 和统计量。它是理解数据集的第一入口。

In [ ]:
metadata = None

if RUN_DATASET:
    from pprint import pprint
    from lerobot.datasets import LeRobotDatasetMetadata

    metadata = LeRobotDatasetMetadata(DATASET_REPO)
    print(f"total_episodes = {metadata.total_episodes}")
    print(f"total_frames   = {metadata.total_frames}")
    print(f"fps            = {metadata.fps}")
    print(f"camera_keys    = {metadata.camera_keys}")
    print("\nfeatures:")
    pprint(metadata.features)
else:
    print("已跳过 Hub 读取：把 RUN_DATASET 改为 True 后重新运行本 Cell。")

## 2. 读取一个 frame

不传 `delta_timestamps` 时，`dataset[0]` 可以先理解为“第 0 个时间点的样本”。返回值是一个字典：图片、state、action、task 等字段都在里面。

In [ ]:
dataset = None
sample = None

if RUN_DATASET:
    from lerobot.datasets import LeRobotDataset

    # episodes=[0] 只选择第 0 个 episode，但 dataset[0] 仍然是这个 episode 中的一个 frame。
    dataset = LeRobotDataset(DATASET_REPO, episodes=EPISODES)
    sample = dataset[0]

    print(f"num_episodes = {dataset.num_episodes}")
    print(f"num_frames   = {dataset.num_frames}")
    print("\nframe fields and shapes:")
    for key, value in sample.items():
        shape = getattr(value, "shape", None)
        dtype = getattr(value, "dtype", None)
        print(f"{key}: type={type(value).__name__}, shape={shape}, dtype={dtype}")
else:
    print("已跳过 frame 读取。")

## 3. 加入时间窗口：历史图片/state 和未来 action

`delta_timestamps` 的单位是秒，都是相对于当前 frame 的偏移。这里用 `1 / fps` 构造合法的 frame 间隔，避免硬编码时间和数据集 fps 不一致。

In [ ]:
window_dataset = None
window_sample = None

if RUN_DATASET:
    import torch
    from lerobot.datasets import LeRobotDataset

    dt = 1.0 / metadata.fps
    camera_key = metadata.camera_keys[0]
    delta_timestamps = {
        # 当前时刻之前两个 frame、之前一个 frame、当前 frame。
        camera_key: [-(HISTORY_STEPS - 1) * dt, -dt, 0.0],
        "observation.state": [-(HISTORY_STEPS - 1) * dt, -dt, 0.0],
        # 当前 action 加上未来 ACTION_STEPS - 1 个动作，形成 action chunk。
        "action": [i * dt for i in range(ACTION_STEPS)],
    }

    window_dataset = LeRobotDataset(
        DATASET_REPO,
        episodes=EPISODES,
        delta_timestamps=delta_timestamps,
    )
    window_sample = window_dataset[0]

    print(f"fps={metadata.fps}, dt={dt:.4f}s")
    print(f"{camera_key}: {tuple(window_sample[camera_key].shape)}")
    print(f"observation.state: {tuple(window_sample['observation.state'].shape)}")
    print(f"action: {tuple(window_sample['action'].shape)}")
else:
    print("已跳过时间窗口读取。")

## 4. DataLoader：从一个样本到一个 batch

如果单个时间窗口图片是 `[T,C,H,W]`，DataLoader 加上 batch 维后就是 `[B,T,C,H,W]`。action 同理从 `[T,A]` 变成 `[B,T,A]`。

In [ ]:
if RUN_DATASET:
    loader = torch.utils.data.DataLoader(
        window_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,  # 初学者先用 0，确认逻辑后再增加 worker
    )
    batch = next(iter(loader))
    print(f"{camera_key}: {tuple(batch[camera_key].shape)}")
    print(f"observation.state: {tuple(batch['observation.state'].shape)}")
    print(f"action: {tuple(batch['action'].shape)}")
else:
    print("已跳过 DataLoader。")

## 检查

打开数据后，至少检查：

- camera Tensor 是否是 `[C,H,W]` 或 `[T,C,H,W]`；
- `observation.state` 的最后一维是否为 8；
- `action` 的最后一维是否为 7；
- task 是否是自然语言字符串；
- DataLoader 是否只增加 batch 维，而没有改变 feature 的最后一维。

## 下一步

读懂这个 Notebook 后，再去读 `02_学习Pi05_LoRA微调.ipynb`。下一步不要马上改模型；先用同一份数据和同一个 checkpoint 跑出 baseline。